# Predicción del consumo eléctrico de viviendas
## Práctica de Sistemas de Aprendizaje Automático
Autor: Fernando Calles Ramos  
Fecha de ejecución: completar al realizar la práctica.

Estado: cuaderno preparado, pendiente de ejecutar con el CSV real. No contiene resultados inventados.

Objetivo: predecir `consumo_mensual_kwh` comparando cuatro algoritmos con un conjunto reducido de 4 variables y otro ampliado de 7 variables. Se realizan ocho entrenamientos y se reserva el 30 % de los registros cribados para test.

Ejecuta las celdas en orden. En la selección de variables tendrás que introducir índices y justificar las columnas reales del CSV. No elijas variables calculadas a partir del consumo objetivo (por ejemplo, una factura calculada directamente a partir de esos mismos kWh).

## 1. Bibliotecas y reproducibilidad
Se conserva una semilla común para repetir la partición y las operaciones aleatorias. No se realiza búsqueda de hiperparámetros: se utilizan configuraciones base, fijando únicamente `random_state` donde procede. Esta práctica nueva no exige mantener estrictamente todos los parámetros por defecto.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import platform
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

SEMILLA = 42
OBJETIVO = 'consumo_mensual_kwh'
SALIDA = Path('resultados_consumo')
SALIDA.mkdir(exist_ok=True)
fecha = datetime.now(ZoneInfo('Europe/Madrid')).isoformat(timespec='seconds')
print('Fecha y hora de ejecución (Madrid):', fecha)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)

## 2. Carga e inspección
El enunciado describe 30.000 registros y 24 columnas. Comprobamos las dimensiones reales, tipos, nulos y ejemplos. En Colab se abrirá el selector si el CSV no está en el directorio de trabajo. La fecha impresa ayuda a documentar la ejecución, pero no sustituye la captura de Colab junto al reloj del sistema que pide el enunciado.

In [ ]:
ruta = Path('dataset_consumo_electrico_30000.csv')
if not ruta.exists():
    try:
        from google.colab import files
        subidos = files.upload()
        candidatos = [nombre for nombre in subidos if nombre.lower().endswith('.csv')]
        if len(candidatos) != 1:
            raise ValueError('Sube exactamente un archivo CSV.')
        ruta = Path(candidatos[0])
    except ImportError:
        raise FileNotFoundError('Coloca el CSV junto al cuaderno antes de continuar.')

df_original = pd.read_csv(ruta, sep=None, engine='python')
df_original.columns = df_original.columns.str.strip()
if df_original.columns.duplicated().any():
    raise ValueError('Hay nombres de columna repetidos. Revisa el encabezado.')
if OBJETIVO not in df_original.columns:
    raise ValueError(f'No aparece {OBJETIVO}. Revisa el separador y los encabezados.')
print('Dimensiones reales:', df_original.shape)
if df_original.shape != (30000, 24):
    print('Aviso: las dimensiones difieren de las indicadas en el enunciado.')
display(df_original.head())
df_original.info()
display(df_original.describe(include='all').T)
inventario = pd.DataFrame({
    'tipo': df_original.dtypes.astype(str),
    'nulos': df_original.isna().sum(),
    'valores_distintos': df_original.nunique()
})
display(inventario)
inventario.to_csv(SALIDA / 'inventario_columnas.csv', encoding='utf-8-sig')

## 3. Cribado inicial
Se normalizan blancos y textos vacíos; los infinitos se consideran ausentes. Se eliminan duplicados exactos de las columnas originales y registros sin un objetivo numérico válido o con consumo negativo. Se conserva el consumo cero: no se ha demostrado que sea incorrecto.

No se eliminan consumos altos por ser altos. Para descartarlos haría falta justificar un error o una imposibilidad física. Los nulos de los predictores se imputarán después usando únicamente el entrenamiento. No se sustituyen etiquetas objetivo ausentes por medias ni medianas.

Si el CSV usa coma decimal, revisa su formato y adapta la carga antes de continuar: no aceptes como correctas conversiones masivas a nulo.

In [ ]:
df = df_original.copy()
for c in df.select_dtypes(include=['object', 'string']).columns:
    df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)
df = df.replace(r'^\s*$', np.nan, regex=True).replace([np.inf, -np.inf], np.nan)
antes = len(df)
df = df.drop_duplicates()
duplicados = antes - len(df)
objetivo_convertido = pd.to_numeric(df[OBJETIVO], errors='coerce')
fallos_conversion = df[OBJETIVO].notna() & objetivo_convertido.isna()
print('Valores del objetivo no convertibles:', int(fallos_conversion.sum()))
if fallos_conversion.any():
    display(df.loc[fallos_conversion, [OBJETIVO]].head(10))
    raise ValueError('Revisa el formato del objetivo antes de eliminar registros.')
df[OBJETIVO] = objetivo_convertido
sin_objetivo = int(df[OBJETIVO].isna().sum())
negativos = int((df[OBJETIVO] < 0).sum())
df = df.loc[df[OBJETIVO].notna() & (df[OBJETIVO] >= 0)].reset_index(drop=True)
cribado = pd.DataFrame({
    'concepto': ['Filas originales', 'Duplicados exactos', 'Sin objetivo',
                 'Consumos negativos', 'Filas conservadas'],
    'cantidad': [len(df_original), duplicados, sin_objetivo, negativos, len(df)]
})
display(cribado)
cribado.to_csv(SALIDA / 'resumen_cribado.csv', index=False, encoding='utf-8-sig')
if len(df) < 20:
    raise ValueError('Demasiado pocos registros; revisa la lectura del CSV.')
if 'id_registro' in df and df['id_registro'].duplicated().any():
    print('ATENCIÓN: hay identificadores repetidos. Revisa si son observaciones de una misma vivienda.')
    raise ValueError('Resuelve los identificadores repetidos antes de continuar para evitar fuga entre particiones.')

## 4. Dos conjuntos de variables
Se eligen 4 variables para el conjunto reducido y se añaden 3 para el ampliado. Así comparamos 4 frente a 7 variables originales, dentro de los límites del enunciado. La codificación categórica puede producir más columnas internas, pero no cambia el número de variables originales elegidas.

Criterios propuestos (solo si existen columnas equivalentes en el CSV):
- Reducido: superficie de la vivienda, número de ocupantes, calefacción eléctrica y uso de aire acondicionado.
- Ampliado: las anteriores más aislamiento, temperatura exterior y cantidad o uso de electrodomésticos.

Son hipótesis a comprobar, no nombres conocidos del archivo ni relaciones ya demostradas. Si no existen, escoge otras variables reales y justifícalas. Evita identificadores, direcciones únicas, el propio objetivo y datos derivados de él. No selecciones variables examinando su MAE en test.

Introduce los índices que aparecen abajo, por ejemplo `0,2,4,6`. Después explica cada elección con su significado real. Revisa especialmente columnas numéricas guardadas como texto y códigos numéricos que realmente representan categorías.

In [ ]:
candidatas = [c for c in df.columns if c not in [OBJETIVO, 'id_registro']]
for i, c in enumerate(candidatas):
    print(f'{i}: {c} ({df[c].dtype})')

def elegir_indices(mensaje, cantidad, excluidas=None):
    indices = [int(v.strip()) for v in input(mensaje).split(',')]
    if len(indices) != cantidad or len(set(indices)) != cantidad:
        raise ValueError(f'Debes elegir {cantidad} índices distintos.')
    if any(i < 0 or i >= len(candidatas) for i in indices):
        raise ValueError('Índice fuera de rango.')
    nombres = [candidatas[i] for i in indices]
    if set(nombres) & set(excluidas or []):
        raise ValueError('Las variables adicionales no deben repetir las reducidas.')
    return nombres

variables_reducidas = elegir_indices('Índices de las 4 variables reducidas: ', 4)
extras = elegir_indices('Índices de 3 variables adicionales: ', 3, variables_reducidas)
variables_ampliadas = variables_reducidas + extras
conjuntos = {'Reducido (4)': variables_reducidas, 'Ampliado (7)': variables_ampliadas}

justificaciones = []
for c in variables_ampliadas:
    razon = input(f'Justifica {c} y su posible relación con el consumo: ').strip()
    if not razon:
        raise ValueError('La justificación no puede quedar vacía.')
    justificaciones.append({'variable': c,
        'conjunto': 'Ambos' if c in variables_reducidas else 'Solo ampliado',
        'justificacion': razon})
seleccion = pd.DataFrame(justificaciones)
display(seleccion)
seleccion.to_csv(SALIDA / 'seleccion_justificada.csv', index=False, encoding='utf-8-sig')

codigos = input('Columnas elegidas que son códigos categóricos numéricos, separadas por comas (Enter si ninguna): ').strip()
categoricas_manuales = [c.strip() for c in codigos.split(',') if c.strip()]
if not set(categoricas_manuales).issubset(variables_ampliadas):
    raise ValueError('Solo puedes indicar columnas elegidas.')
for c in categoricas_manuales:
    df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan)

## 5. División común: 70 % entrenamiento y 30 % test
Se divide una sola vez el índice de filas para que todos los modelos y ambos conjuntos se evalúen sobre las mismas viviendas. Con 30.000 filas conservadas serían 21.000 y 9.000. Si el cribado elimina registros, el reparto se aplica a los restantes.

Esta metodología presupone observaciones independientes. Si hay varias observaciones de una misma vivienda o estructura temporal que deba respetarse, hay que adaptar la partición por grupos o fechas antes de interpretar los resultados.

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.30, random_state=SEMILLA)
train_y = df.loc[train_idx, OBJETIVO]
test_y = df.loc[test_idx, OBJETIVO]
print('Entrenamiento:', len(train_idx), '| Test:', len(test_idx))
for c in variables_ampliadas:
    if df.loc[train_idx, c].notna().sum() == 0:
        raise ValueError(f'{c} no tiene datos en entrenamiento: elige otra variable.')

## 6. Preprocesamiento sin fuga de información
Se sustituyen los nulos numéricos por la mediana del entrenamiento y los categóricos por la categoría más frecuente. Las categorías se convierten con OneHotEncoder y se toleran categorías nuevas en test. La regresión lineal incorpora estandarización; los árboles no la necesitan.

El Pipeline ajusta imputación, codificación y escalado exclusivamente en `fit(train_x, train_y)`. No se ajusta el preprocesamiento con todo el CSV antes de la división.

In [ ]:
def crear_preprocesador(train_x, escalar=False):
    numericas = train_x.select_dtypes(include=np.number).columns.tolist()
    categoricas = [c for c in train_x.columns if c not in numericas]
    pasos_num = [('imputar', SimpleImputer(strategy='median'))]
    if escalar:
        pasos_num.append(('escalar', StandardScaler()))
    transformadores = []
    if numericas:
        transformadores.append(('numericas', Pipeline(pasos_num), numericas))
    if categoricas:
        pasos_cat = Pipeline([
            ('imputar', SimpleImputer(strategy='most_frequent')),
            ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        transformadores.append(('categoricas', pasos_cat, categoricas))
    return ColumnTransformer(transformadores, remainder='drop')

## 7. Investigación y entrenamiento de cuatro modelos
1. Regresión lineal: referencia sencilla que ajusta una combinación lineal de los predictores. Permite comprobar si una relación simple es suficiente.
2. Árbol de decisión: aprende reglas mediante divisiones de los datos; capta umbrales e interacciones y puede sobreajustar.
3. Random Forest: combina árboles entrenados con aleatoriedad y promedia sus predicciones. Permite contrastar un árbol individual con un ensamble.
4. Gradient Boosting: construye árboles de forma secuencial para mejorar progresivamente la predicción. Permite comparar dos estrategias diferentes de ensamble.

Se calculan MAE de entrenamiento y de test, y tiempos orientativos. Un MAE de entrenamiento muy inferior al de test es una señal compatible con sobreajuste, no una prueba por sí sola.

Se añade una referencia de mediana (DummyRegressor) que no cuenta como uno de los cuatro algoritmos solicitados. Permite comprobar si los modelos mejoran una predicción constante.

In [ ]:
def crear_modelos():
    return {
        'Regresion lineal': LinearRegression(),
        'Arbol de decision': DecisionTreeRegressor(random_state=SEMILLA),
        'Random Forest': RandomForestRegressor(random_state=SEMILLA),
        'Gradient Boosting': GradientBoostingRegressor(random_state=SEMILLA)
    }

resultados, predicciones = [], []
modelos_entrenados = {}
for nombre_conjunto, columnas in conjuntos.items():
    train_x = df.loc[train_idx, columnas]
    test_x = df.loc[test_idx, columnas]
    for nombre_modelo, modelo in crear_modelos().items():
        pipeline = Pipeline([
            ('preprocesar', crear_preprocesador(train_x, nombre_modelo == 'Regresion lineal')),
            ('modelo', modelo)
        ])
        inicio = time.perf_counter()
        pipeline.fit(train_x, train_y)
        tiempo_entrenamiento = time.perf_counter() - inicio
        inicio = time.perf_counter()
        pred_test = pipeline.predict(test_x)
        tiempo_prediccion = time.perf_counter() - inicio
        pred_train = pipeline.predict(train_x)
        resultados.append({
            'Conjunto': nombre_conjunto, 'Modelo': nombre_modelo,
            'Variables': len(columnas),
            'MAE_train_kWh': mean_absolute_error(train_y, pred_train),
            'MAE_test_kWh': mean_absolute_error(test_y, pred_test),
            'Entrenamiento_s': tiempo_entrenamiento,
            'Prediccion_test_s': tiempo_prediccion
        })
        modelos_entrenados[(nombre_conjunto, nombre_modelo)] = pipeline
        predicciones.append(pd.DataFrame({
            'fila': test_idx, 'Conjunto': nombre_conjunto, 'Modelo': nombre_modelo,
            'Real_kWh': test_y.to_numpy(), 'Predicho_kWh': pred_test,
            'Error_absoluto_kWh': np.abs(test_y.to_numpy() - pred_test)
        }))
        print(nombre_conjunto, '|', nombre_modelo, '| MAE:', round(resultados[-1]['MAE_test_kWh'], 3))

tabla_resultados = pd.DataFrame(resultados).sort_values('MAE_test_kWh').reset_index(drop=True)
tabla_predicciones = pd.concat(predicciones, ignore_index=True)
base = DummyRegressor(strategy='median')
base.fit(np.zeros((len(train_y), 1)), train_y)
mae_base = mean_absolute_error(test_y, base.predict(np.zeros((len(test_y), 1))))
tabla_resultados['Mejora_vs_mediana_pct'] = (
    100 * (mae_base - tabla_resultados['MAE_test_kWh']) / mae_base if mae_base > 0 else np.nan
)
print('MAE de la referencia de mediana:', mae_base, 'kWh')
display(tabla_resultados.round(3))
tabla_resultados.to_csv(SALIDA / 'resultados_modelos.csv', index=False, encoding='utf-8-sig')
tabla_predicciones.to_csv(SALIDA / 'predicciones_test.csv', index=False, encoding='utf-8-sig')

## 8. Influencia de las variables
La mejora se calcula como MAE reducido menos MAE ampliado. Un valor positivo indica que añadir las tres variables redujo el error. Un valor negativo indica que empeoró. Más variables no garantizan mejores predicciones.

In [ ]:
comparacion = tabla_resultados.pivot(index='Modelo', columns='Conjunto', values='MAE_test_kWh')
comparacion['Reduccion_MAE_kWh'] = comparacion['Reducido (4)'] - comparacion['Ampliado (7)']
comparacion['Mejora_pct'] = 100 * comparacion['Reduccion_MAE_kWh'] / comparacion['Reducido (4)'].replace(0, np.nan)
display(comparacion.round(3))
comparacion.to_csv(SALIDA / 'comparacion_variables.csv', encoding='utf-8-sig')

## 9. Gráficos e interpretación de las predicciones
Se incluyen MAE por algoritmo y conjunto, valores reales frente a predichos y distribución de errores (real menos predicho). Por encima de la diagonal de igualdad el modelo sobreestima; por debajo subestima. En el histograma, un error positivo indica subestimación y uno negativo sobreestimación.

Las figuras del candidato con menor MAE son descriptivas de esta comparación; no constituyen una segunda evaluación independiente.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
comparacion[['Reducido (4)', 'Ampliado (7)']].plot.bar(ax=ax)
ax.set_ylabel('MAE de test (kWh)')
ax.set_title('Comparación de algoritmos y conjuntos de variables')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(SALIDA / 'comparacion_mae.png', dpi=160)
plt.show()

mejor = tabla_resultados.iloc[0]
mejores_preds = tabla_predicciones.loc[
    (tabla_predicciones['Conjunto'] == mejor['Conjunto']) &
    (tabla_predicciones['Modelo'] == mejor['Modelo'])
].copy()
display(mejores_preds.head(10).round(3))
mejores_preds.head(10).to_csv(SALIDA / 'ejemplos_predicciones.csv', index=False, encoding='utf-8-sig')
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(mejores_preds['Real_kWh'], mejores_preds['Predicho_kWh'], alpha=0.25, s=8)
lim_inf = min(mejores_preds['Real_kWh'].min(), mejores_preds['Predicho_kWh'].min())
lim_sup = max(mejores_preds['Real_kWh'].max(), mejores_preds['Predicho_kWh'].max())
axes[0].plot([lim_inf, lim_sup], [lim_inf, lim_sup], color='red', linestyle='--')
axes[0].set(xlabel='Consumo real (kWh)', ylabel='Consumo predicho (kWh)', title='Real frente a predicho')
errores = mejores_preds['Real_kWh'] - mejores_preds['Predicho_kWh']
axes[1].hist(errores, bins=40, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set(xlabel='Real - predicho (kWh)', ylabel='Número de viviendas', title='Distribución de errores')
fig.tight_layout()
fig.savefig(SALIDA / 'diagnostico_predicciones.png', dpi=160)
plt.show()
print('Predicciones negativas del candidato:', int((mejores_preds['Predicho_kWh'] < 0).sum()))

## 10. Respuestas y conclusiones basadas en resultados reales
MAE = media del valor absoluto de la diferencia entre consumo real y predicho. Se expresa en kWh, no en porcentaje. No significa que todas las viviendas tengan ese mismo error.

La celda genera un borrador con resultados calculados. Completa después el análisis de los gráficos y de las variables concretas seleccionadas. El menor MAE es un criterio inicial, no la única condición para desplegar un modelo.

Limitación: al comparar ocho alternativas y elegir con el mismo test, su MAE ganador puede ser optimista como estimación del rendimiento del modelo seleccionado. Antes de producción conviene seleccionar con validación cruzada dentro del entrenamiento y evaluar una sola vez la opción fijada en un test independiente. No se debe retocar repetidamente el modelo mirando este test.

In [ ]:
lineas = [
    '# Resultados de la ejecución',
    f'Fecha: {fecha}',
    f'Filas conservadas: {len(df)}. Entrenamiento: {len(train_idx)}. Test: {len(test_idx)}.',
    '',
    '## Algoritmo con menor MAE',
    f"En esta partición, el menor MAE de test lo obtuvo {mejor['Modelo']} con el conjunto {mejor['Conjunto']}: {mejor['MAE_test_kWh']:.3f} kWh.",
    'Ese valor es el promedio de los errores absolutos por vivienda, no un porcentaje ni un error máximo.',
    f'La referencia que siempre predice la mediana del entrenamiento obtuvo {mae_base:.3f} kWh.',
    '',
    '## Influencia de la selección de variables'
]
for nombre, fila in comparacion.iterrows():
    cambio = fila['Reduccion_MAE_kWh']
    efecto = 'redujo' if cambio >= 0 else 'aumentó'
    lineas.append(f'- {nombre}: ampliar de 4 a 7 variables {efecto} el MAE en {abs(cambio):.3f} kWh.')
lineas += [
    '', '## Candidato para producción',
    f"Por exactitud en esta comparación, el candidato inicial es {mejor['Modelo']} con {mejor['Conjunto']}.",
    'La elección definitiva requiere confirmar estabilidad con validación adicional, revisar errores por tipos de vivienda, disponibilidad de variables, coste de mantenimiento, tiempo de inferencia y posibles predicciones negativas.',
    'Si otro modelo obtiene un error muy parecido con menor complejidad o menos variables, puede ser preferible. Los tiempos medidos aquí son orientativos de una ejecución, no un benchmark definitivo.',
    '', '## Límites y tareas pendientes de la memoria',
    '- Incorporar las justificaciones concretas de las siete variables.',
    '- Explicar las diferencias observadas entre MAE de entrenamiento y test.',
    '- Comentar los gráficos y las diez predicciones de ejemplo.',
    '- Incluir capturas auténticas de Google Colab con fecha y hora del sistema.',
    '- No presentar esta comparación de test como una evaluación externa independiente de la selección.'
]
texto = '\n\n'.join(lineas)
print(texto)
(SALIDA / 'conclusiones_calculadas.md').write_text(texto, encoding='utf-8')
configuracion = {'fecha': fecha, 'semilla': SEMILLA, 'objetivo': OBJETIVO,
                 'conjuntos': conjuntos, 'sklearn': sklearn.__version__,
                 'filas_train': len(train_idx), 'filas_test': len(test_idx)}
(SALIDA / 'configuracion.json').write_text(json.dumps(configuracion, indent=2, ensure_ascii=False), encoding='utf-8')

## 11. Descarga de resultados
El ZIP incluye las tablas, figuras, selección justificada, configuración y conclusiones calculadas de esta ejecución. Guarda también el cuaderno con sus salidas y las capturas reales. No contiene un modelo serializado ni sustituye la memoria técnica.

In [ ]:
import shutil
archivo_zip = shutil.make_archive('resultados_consumo', 'zip', root_dir=SALIDA)
print('Resultados exportados correctamente.')
try:
    from google.colab import files
    files.download(archivo_zip)
except ImportError:
    print('El ZIP está en el directorio de trabajo del cuaderno.')

## 12. Webgrafía
- Enunciado facilitado: Proyecto_MachineLearning_Regresion.pdf.
- Ejemplo docente: ejemplo_machine_learning.ipynb.
- Preprocesamiento mixto: https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html
- Regresión lineal: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- Árbol de regresión: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html
- Random Forest: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- Gradient Boosting: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html
- MAE: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html

Registra en la memoria la fecha real de consulta de las fuentes que utilices.